# InaTRC × YOLO26n — Full Reproducibility & W&B Audit (Baseline v2)

**Purpose:** A rigorously documented **non-KD baseline**, with an epoch-by-epoch W&B history **and** a versioned model artifact for **every training epoch**. A separate final artifact contains `best.pt`, `last.pt`, complete histories, evaluation reports and plots.

**Protocol kept from the successful baseline:** YOLO26n pretrained, seed=42, 640 px, batch=16, epochs≤100, patience=25, AdamW lr0=0.001, weight_decay=0.0005, warmup=3, AMP on, close_mosaic=10. Validation only: **test split remains untouched**.

**IMPORTANT**
- Kaggle: GPU accelerator **ON**, Internet **ON**, input dataset InaTRC attached, Secret `WANDB_API_KEY` enabled.
- Run cells from top to bottom once in a fresh Kaggle session. Changing parameters creates a **different experimental protocol**; do not merge those results with older baselines.
- 100 checkpoint uploads can consume substantial W&B bandwidth/storage and training time. Default behavior waits for upload success then deletes only the redundant local `epochNN.pt` (the checkpoint remains available via W&B artifact). `best.pt`/`last.pt` are always retained locally.
- W&B **`latest` is an artifact alias**, not the `last.pt` file. `best` identifies the version associated with the best validation fitness.
- Dataset leakage audit here checks **exact SHA-256 duplicates** only, **not** visually near-identical CCTV frames. Conduct perceptual/group leakage audit before final evaluation.
- Code and metric semantics documented from official [W&B Artifacts](https://docs.wandb.ai/models/artifacts/create-a-new-artifact-version), [W&B Runs](https://docs.wandb.ai/ref/python/experiments/run/), and [Ultralytics callbacks](https://docs.ultralytics.com/usage/callbacks) documentation.

## 0 — Install exact dependencies; disable **native** Ultralytics W&B integration to prevent duplicate runs and premature finish

In [ ]:
%pip install -q "ultralytics==8.4.155" "wandb==0.28.1" "faster-coco-eval==1.6.7" "thop" "pyyaml" "pandas" "matplotlib" "pillow" "nbformat"
# We implement ONE explicit W&B run + callbacks ourselves; native callback would finish() earlier.
!yolo settings wandb=False

## 1 — Imports, locked protocol, environment

In [ ]:
import gc, hashlib, json, os, platform, re, sys, time, zipfile
from collections import defaultdict
from copy import deepcopy
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from IPython.display import display, FileLink
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

assert ultralytics.__version__ == "8.4.155", "Restart kernel after pip install."
assert wandb.__version__ == "0.28.1", "Restart kernel after pip install."
assert settings.get("wandb") is False, "Native W&B integration must be OFF in this notebook."
assert torch.cuda.is_available(), "Enable GPU accelerator in Kaggle settings."

CFG = {
    "seed": 42,
    "dataset_name": "InaTRC",
    "dataset_root": None,           # Explicit path if several compatible datasets found.
    "class_names": [
        "Kelas-1 (Non-truk)", "Kelas-2 (Truk 2 sumbu)",
        "Kelas-3 (Truk 3 sumbu)", "Kelas-4 (Truk 4 sumbu)",
        "Kelas-5 (Truk >=5 sumbu)",
    ],
    "student_model": "yolo26n.pt",
    "imgsz": 640, "batch": 16, "epochs": 100, "patience": 25,
    "optimizer": "AdamW", "lr0": 0.001, "weight_decay": 0.0005,
    "warmup_epochs": 3.0, "amp": True, "workers": 2,
    "cache": False, "close_mosaic": 10, "flipud": 0.0,
    "mixup": 0.0, "cutmix": 0.0, "deterministic": True,
    "compile": False, "save_period": 1, "plots": True, "val": True,
    "wandb_entity": "ilhamrafiqin-brawijaya-university",
    "wandb_project": "skripsi-inatrc-yolo26",
    "wandb_group": "baseline-audited-v2",
    "experiment_version": "v2-full-wandb",
    "upload_checkpoint_each_epoch": True,
    "delete_local_epoch_after_upload": True,
}
CFG["run_name"] = f"inatrc-yolo26n-baseline-seed{CFG['seed']}-{CFG['experiment_version']}"

PROTOCOL_KEYS = [
    "student_model", "imgsz", "batch", "epochs", "patience", "optimizer",
    "lr0", "weight_decay", "warmup_epochs", "amp", "workers", "cache",
    "close_mosaic", "flipud", "mixup", "cutmix", "deterministic",
    "compile", "save_period", "plots", "val", "seed"
]
assert CFG["save_period"] == 1, "Every-epoch artifact requires checkpoint each epoch."
assert CFG["upload_checkpoint_each_epoch"] is True
assert CFG["student_model"] == "yolo26n.pt"
assert CFG["epochs"] == 100 and CFG["batch"] == 16 and CFG["imgsz"] == 640

IN_KAGGLE = Path('/kaggle/working').exists()
ROOT = Path('/kaggle/working' if IN_KAGGLE else '.').resolve() / "inatrc_yolo26"
WORK_ROOT = ROOT / CFG["run_name"]
REPORTS = WORK_ROOT / "reports"
LOCAL_PROJECT_DIR = WORK_ROOT / "runs"
for p in (REPORTS, LOCAL_PROJECT_DIR): p.mkdir(parents=True, exist_ok=True)

DEVICE = 0
ENV = {
    "platform": "kaggle" if IN_KAGGLE else "other",
    "python": platform.python_version(),
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "ultralytics": ultralytics.__version__, "wandb": wandb.__version__,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "gpu_vram_gib": round(torch.cuda.get_device_properties(DEVICE).total_memory / 2**30, 3),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
init_seeds(CFG["seed"], deterministic=CFG["deterministic"])
(REPORTS / "environment.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")
(REPORTS / "experiment_config.json").write_text(json.dumps(CFG, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(ENV, indent=2))
print("Experiment root:", WORK_ROOT)

## 2 — Kaggle W&B secret (do not paste the API key)

In [ ]:
def get_secret(name):
    if os.getenv(name):
        return os.environ[name]
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

WANDB_API_KEY = get_secret("WANDB_API_KEY")
assert WANDB_API_KEY, "Add and enable Kaggle Secret WANDB_API_KEY."
wandb.login(key=WANDB_API_KEY, relogin=True)
del WANDB_API_KEY
print("W&B:", CFG["wandb_entity"], "/", CFG["wandb_project"])

## 3 — Discover InaTRC and validate annotation schema

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_METADATA = {"classes.txt", "_classes.txt"}
SPLITS = ("train", "val", "test")

def is_root(root):
    return all((root / s / "images").is_dir() and (root / s / "labels").is_dir() for s in SPLITS)

def discover_dataset():
    if CFG["dataset_root"]:
        path = Path(CFG["dataset_root"]).expanduser().resolve()
        assert is_root(path), f"Invalid dataset root: {path}"
        return path
    root = Path('/kaggle/input') if IN_KAGGLE else Path('.')
    candidates = {p.parent.resolve() for p in root.rglob('train') if is_root(p.parent)}
    if is_root(root): candidates.add(root.resolve())
    assert len(candidates) == 1, f"Expected ONE InaTRC root, found: {sorted(candidates)}. Set CFG['dataset_root']."
    return next(iter(candidates))

DATA_ROOT = discover_dataset()
SPLIT_IMAGES = {s: DATA_ROOT / s / "images" for s in SPLITS}
SPLIT_LABELS = {s: DATA_ROOT / s / "labels" for s in SPLITS}
print("InaTRC:", DATA_ROOT)

In [ ]:
audit, per_cls, boxes, duplicate = [], [], [], []
sha_to_images = defaultdict(list)
fingerprint = hashlib.sha256()
all_images = []

for split in SPLITS:
    images = sorted(p for p in SPLIT_IMAGES[split].iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXT)
    labels = sorted(p for p in SPLIT_LABELS[split].iterdir() if p.is_file() and p.suffix.lower() == '.txt'
                    and p.name.lower() not in LABEL_METADATA)
    by_img, by_label = {p.stem:p for p in images}, {p.stem:p for p in labels}
    assert len(by_img) == len(images), f"{split} has duplicate image stems."
    assert len(by_label) == len(labels), f"{split} has duplicate label stems."
    assert set(by_img) == set(by_label), (
        f"{split}: missing labels={sorted(set(by_img)-set(by_label))[:5]}, "
        f"orphan labels={sorted(set(by_label)-set(by_img))[:5]}"
    )
    n_boxes, n_empty, per_class_counts = 0, 0, np.zeros(5, dtype=np.int64)
    for stem, img in sorted(by_img.items()):
        label = by_label[stem]
        img_sha = hashlib.sha256(img.read_bytes()).hexdigest()
        label_bytes = label.read_bytes()
        fingerprint.update(f"{split}/{img.name}\n{img_sha}\n{label.name}\n".encode())
        fingerprint.update(label_bytes)
        sha_to_images[img_sha].append((split, img.name))
        all_images.append(img)
        lines = label_bytes.decode('utf-8-sig').strip().splitlines()
        if not lines: n_empty += 1
        for line_idx, line in enumerate(lines, start=1):
            v = line.split()
            assert len(v) == 5, f"{split}/{label.name}:{line_idx} must be YOLO box (5 values), got {len(v)}"
            nums = np.asarray([float(k) for k in v], dtype=float)
            assert np.isfinite(nums).all(), f"Non-finite label: {label}"
            cl, x, y, w, h = nums
            assert cl.is_integer() and 0 <= cl < 5, f"Invalid class: {label}"
            assert (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                    and x - w/2 >= -1e-4 and y - h/2 >= -1e-4
                    and x + w/2 <= 1+1e-4 and y + h/2 <= 1+1e-4), f"Invalid bbox: {label}"
            cls = int(cl)
            per_class_counts[cls] += 1
            n_boxes += 1
            boxes.append({"split":split, "class_id":cls, "class_name":CFG['class_names'][cls],
                          "normalized_area":float(w*h), "normalized_width":float(w), "normalized_height":float(h)})
    audit.append({"split":split, "images":len(images), "labels":len(labels),
                  "boxes":n_boxes, "empty_labels":n_empty})
    for cls, num in enumerate(per_class_counts):
        per_cls.append({"split":split, "class_id":cls, "class_name":CFG['class_names'][cls], "boxes":int(num)})

for sha, occurrences in sha_to_images.items():
    if len({s for s, _ in occurrences}) > 1:
        duplicate.append({"sha256":sha, "occurrences":str(occurrences)})

AUDIT = pd.DataFrame(audit)
CLASS_COUNTS = pd.DataFrame(per_cls)
BOXES = pd.DataFrame(boxes)
DUPLICATES = pd.DataFrame(duplicate, columns=["sha256", "occurrences"])
assert int(AUDIT.images.sum()) == 3250, f"Expected 3250 images, got {AUDIT.images.sum()}"
DATA_FINGERPRINT = fingerprint.hexdigest()

for file, df in (("dataset_audit.csv",AUDIT),("class_distribution.csv",CLASS_COUNTS),
                 ("bbox_distribution.csv",BOXES),("cross_split_exact_duplicates.csv",DUPLICATES)):
    df.to_csv(REPORTS / file, index=False)
(REPORTS / "dataset_fingerprint.json").write_text(json.dumps({
    "sha256": DATA_FINGERPRINT, "source_path":str(DATA_ROOT),
    "note":"SHA-256 image bytes+label bytes+relative split/name; exact duplicate check only",
    "near_duplicate_or_sequence_leakage_checked": False,
    "exact_cross_split_duplicate_groups":len(DUPLICATES)
}, indent=2), encoding='utf-8')

display(AUDIT)
display(CLASS_COUNTS.pivot(index='class_name', columns='split', values='boxes').fillna(0).astype(int))
if len(DUPLICATES):
    print("WARNING: exact image duplicates found across splits; investigate BEFORE final study.")
    display(DUPLICATES.head())
else:
    print("No exact cross-split image duplicates. Temporal/perceptual leakage remains unchecked.")
print("Fingerprint:", DATA_FINGERPRINT)

## 4 — Create dataset YAML and profile raw pretrained model (informational; final profiling happens on best.pt)

In [ ]:
RUNTIME_YAML = WORK_ROOT / "data_runtime.yaml"
yaml_data = {
    "path":str(DATA_ROOT),
    "train":str(SPLIT_IMAGES['train']), "val":str(SPLIT_IMAGES['val']),
    "test":str(SPLIT_IMAGES['test']), "nc":5,
    "names":CFG['class_names']
}
RUNTIME_YAML.write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')

student = YOLO(CFG['student_model'])
student.model.names = dict(enumerate(CFG['class_names']))
PARAMS_INITIAL = int(sum(p.numel() for p in student.model.parameters()))
COMPLEXITY = {"parameters_pretrained":PARAMS_INITIAL,
              "parameters_millions_pretrained":PARAMS_INITIAL / 1e6,
              "GFLOPs_pretrained_estimate":None,
              "GFLOPs_convention":"2 * THOP_MACs / 1e9 on float32 unfused model; estimate"}
try:
    from thop import profile as thop_profile
    temp_model = deepcopy(student.model).float().eval()
    with torch.inference_mode():
        macs, _ = thop_profile(temp_model, inputs=(torch.zeros(1,3,CFG['imgsz'],CFG['imgsz']),), verbose=False)
    COMPLEXITY['GFLOPs_pretrained_estimate'] = float(2*macs/1e9)
except Exception as exc:
    COMPLEXITY['GFLOPs_profile_error'] = repr(exc)
finally:
    if 'temp_model' in globals(): del temp_model
    gc.collect()
(REPORTS/'complexity_pretrained.json').write_text(json.dumps(COMPLEXITY,indent=2),encoding='utf-8')
print(COMPLEXITY)

## 5 — GPU preflight: synthetic detection loss backward at **exact** requested batch

In [ ]:
PREFLIGHT = {"status":"started", "batch":CFG['batch'], "imgsz":CFG['imgsz']}
probe = None
try:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    with torch.random.fork_rng(devices=[DEVICE]):
        probe = deepcopy(student.model).to(DEVICE).train()
        # .pt may be inference-frozen. This preflight bypasses Trainer, which unfreezes during training.
        for p in probe.parameters(): p.requires_grad_(True)
        assert any(p.requires_grad for p in probe.parameters())
        from ultralytics.cfg import get_cfg
        probe.args = get_cfg()
        B = CFG['batch']
        sample = {
            'img':torch.rand(B,3,CFG['imgsz'],CFG['imgsz'],device=DEVICE),
            'batch_idx':torch.arange(B,device=DEVICE),
            'cls':(torch.arange(B,device=DEVICE)%5).float().view(-1,1),
            'bboxes':torch.tensor([[0.5,0.5,0.12,0.12]],device=DEVICE).repeat(B,1),
        }
        with torch.autocast('cuda', enabled=CFG['amp']):
            loss, items = probe(sample)
            total = loss.sum()
        assert torch.isfinite(total).item() and total.requires_grad, 'Preflight loss invalid / no gradient'
        total.backward()
    PREFLIGHT.update(status='passed', loss=float(total.detach()),
                     peak_memory_gib=torch.cuda.max_memory_allocated(DEVICE)/2**30)
except torch.OutOfMemoryError:
    PREFLIGHT['status'] = 'OOM'
    raise RuntimeError('GPU preflight OOM with locked batch=16/imgsz=640; change protocol consistently for all models.')
finally:
    (REPORTS/'gpu_preflight.json').write_text(json.dumps(PREFLIGHT,indent=2),encoding='utf-8')
    for name in ('probe','sample','loss','items','total'):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()
print(PREFLIGHT)

## 6 — Start exactly ONE W&B run; audit tables/config logged at step 0

In [ ]:
assert wandb.run is None, "Active W&B run detected: start a fresh kernel instead of accidentally mixing runs."
WB_RUN = wandb.init(
    entity=CFG['wandb_entity'], project=CFG['wandb_project'], name=CFG['run_name'],
    group=CFG['wandb_group'], job_type='baseline-training',
    tags=['InaTRC','YOLO26n','baseline','fully-audited',f"seed-{CFG['seed']}"],
    config={**CFG,'dataset_root_resolved':str(DATA_ROOT),
            'dataset_fingerprint':DATA_FINGERPRINT,'environment':ENV,
            'complexity_pretrained':COMPLEXITY,'gpu_preflight':PREFLIGHT},
    dir=str(WORK_ROOT), reinit=False
)
assert WB_RUN is not None
WB_RUN.summary['audit/status'] = 'training-started'
WB_RUN.summary['audit/test_evaluated'] = False
WB_RUN.summary['audit/near_duplicate_checked'] = False
WB_RUN.summary['audit/exact_cross_split_duplicate_groups'] = len(DUPLICATES)
WB_RUN.log({
    'epoch':0,
    'dataset/audit_table':wandb.Table(dataframe=AUDIT),
    'dataset/class_distribution':wandb.Table(dataframe=CLASS_COUNTS),
    'dataset/boxes_total':int(AUDIT.boxes.sum()),
    'dataset/images_total':int(AUDIT.images.sum()),
    'dataset/empty_labels_total':int(AUDIT.empty_labels.sum()),
    'model/parameters_pretrained':PARAMS_INITIAL,
}, step=0)
print('W&B RUN:',WB_RUN.url)

## 7 — Explicit W&B callbacks: per-epoch metrics **and** checkpoint artifacts

**Mechanics:** `on_model_save` is fired after Ultralytics saves a checkpoint. With `save_period=1`, it logs `epochN.pt` as a new version of **one W&B model artifact collection**, bearing alias `epoch-001`, `epoch-002`, etc. The **best** alias moves only on best-fitness epochs; W&B **latest** moves every time. The callback waits for the upload to complete before optional local cleanup.

`on_fit_epoch_end` records one scalar history step for every actual training epoch. Ultralytics calls this callback *one additional time* after final best-model validation; we suppress that extra history event using the last saved epoch sentinel.

In [ ]:
RUN_STATE = {
    'last_saved_epoch':0, 'last_logged_epoch':0, 'best_epoch':None,
    'best_fitness':None, 'artifact_versions':[], 'epoch_rows':[],
    'actual_batch':None, 'upload_seconds_total':0.0,
    'per_class_epoch_rows':[], 'epoch_start_perf':None, 'train_val_seconds_last':None,
}
EPOCH_ARTIFACT_NAME = f"inatrc-yolo26n-baseline-epochs-{WB_RUN.id}"

def ensure_baseline(trainer):
    assert trainer.args.batch == CFG['batch'] and trainer.batch_size == CFG['batch']
    assert trainer.args.save_period == 1
    assert trainer.args.optimizer == CFG['optimizer']
    assert trainer.args.compile is False
    assert getattr(trainer.args,'distill_model',None) in (None,'',False)
    assert trainer.args.epochs == CFG['epochs']
    assert wandb.run is WB_RUN
    RUN_STATE['actual_batch'] = int(trainer.batch_size)

def on_epoch_started(trainer):
    RUN_STATE['epoch_start_perf'] = time.perf_counter()

def on_checkpoint_saved(trainer):
    epoch = int(trainer.epoch + 1)
    checkpoint = Path(trainer.wdir) / f"epoch{trainer.epoch}.pt"  # Ultraytics is zero-indexed on disk
    assert checkpoint.exists(), f"Missing epoch checkpoint: {checkpoint}"
    RUN_STATE['train_val_seconds_last'] = (
        time.perf_counter() - RUN_STATE['epoch_start_perf']
        if RUN_STATE['epoch_start_perf'] is not None else None
    )
    fitness = None if trainer.fitness is None else float(trainer.fitness)
    best_fit = None if trainer.best_fitness is None else float(trainer.best_fitness)
    is_best = (fitness is not None and best_fit is not None and fitness == best_fit)
    if is_best:
        RUN_STATE['best_epoch'] = epoch
        RUN_STATE['best_fitness'] = best_fit

    upload_seconds = 0.0
    art_version = None
    if CFG['upload_checkpoint_each_epoch']:
        start = time.perf_counter()
        art = wandb.Artifact(
            name=EPOCH_ARTIFACT_NAME, type='model',
            metadata={"epoch":epoch, "epoch_zero_based":int(trainer.epoch),
                      "fitness":fitness, "best_fitness":best_fit,
                      "is_best_at_save":bool(is_best), "seed":CFG['seed'],
                      "dataset_fingerprint":DATA_FINGERPRINT},
        )
        # Keep the artifact layout stable; each version stores ONE checkpoint.
        art.add_file(str(checkpoint), name='checkpoint.pt')
        aliases = [f"epoch-{epoch:03d}"] + (['best'] if is_best else [])
        logged = WB_RUN.log_artifact(art, aliases=aliases)
        logged.wait()  # Fail loudly if upload isn't confirmed; prevents silently losing checkpoints.
        art_version = logged.version
        upload_seconds = time.perf_counter() - start
        RUN_STATE['artifact_versions'].append(str(art_version))
        if CFG['delete_local_epoch_after_upload']:
            checkpoint.unlink()  # best.pt + last.pt remain locally; artifact is confirmed uploaded.
    RUN_STATE['upload_seconds_total'] += upload_seconds

    row = {
        'epoch':epoch, 'fitness':fitness, 'best_fitness':best_fit,
        'is_best_at_save':int(is_best), 'artifact_collection':EPOCH_ARTIFACT_NAME,
        'artifact_version':art_version, 'artifact_epoch_alias':f"epoch-{epoch:03d}",
        'upload_seconds':upload_seconds,
    }
    RUN_STATE['epoch_rows'].append(row)
    RUN_STATE['last_saved_epoch'] = epoch
    pd.DataFrame(RUN_STATE['epoch_rows']).to_csv(REPORTS/'checkpoint_manifest.csv',index=False)
    WB_RUN.summary['checkpoint/best_epoch'] = RUN_STATE['best_epoch']
    WB_RUN.summary['checkpoint/last_epoch'] = epoch
    WB_RUN.summary['checkpoint/epoch_artifact_collection'] = EPOCH_ARTIFACT_NAME
    WB_RUN.summary['checkpoint/epochs_uploaded'] = len(RUN_STATE['artifact_versions'])


def safe_numeric(x):
    try:
        value = float(x)
        return value if np.isfinite(value) else None
    except (TypeError, ValueError):
        return None

def on_fit_epoch_logged(trainer):
    epoch = int(trainer.epoch + 1)
    # final_eval() emits a callback without on_model_save -- don't log a fake epoch.
    if epoch != RUN_STATE['last_saved_epoch'] or epoch <= RUN_STATE['last_logged_epoch']:
        return
    scalars = {'epoch':epoch}
    scalars.update({k:safe_numeric(v) for k,v in trainer.label_loss_items(trainer.tloss, prefix='train').items()})
    scalars.update({k:safe_numeric(v) for k,v in (trainer.metrics or {}).items()})
    scalars.update({k:safe_numeric(v) for k,v in (trainer.lr or {}).items()})
    scalars.update({
        'fit/fitness':safe_numeric(trainer.fitness),
        'fit/best_fitness':safe_numeric(trainer.best_fitness),
        'fit/is_current_best':int(RUN_STATE['best_epoch'] == epoch),
        'fit/train_val_seconds_excl_artifact_upload':safe_numeric(RUN_STATE['train_val_seconds_last']),
        'fit/epoch_seconds_including_artifact_upload':safe_numeric(getattr(trainer,'epoch_time',None)),
        'system/gpu_memory_allocated_gib':torch.cuda.memory_allocated(DEVICE)/2**30,
        'system/gpu_memory_reserved_gib':torch.cuda.memory_reserved(DEVICE)/2**30,
    })
    # Record *per-class* metrics each epoch, not just the global mean. Box AP thresholds are 0.50:0.05:0.95.
    box_epoch = getattr(getattr(getattr(trainer,'validator',None),'metrics',None),'box',None)
    if box_epoch is not None:
        ids = np.asarray(getattr(box_epoch,'ap_class_index',[]),dtype=int)
        p = np.asarray(getattr(box_epoch,'p',[]),dtype=float)
        r = np.asarray(getattr(box_epoch,'r',[]),dtype=float)
        ap50 = np.asarray(getattr(box_epoch,'ap50',[]),dtype=float)
        ap = np.asarray(getattr(box_epoch,'ap',[]),dtype=float)
        for j,cl in enumerate(ids):
            if j>=len(p) or j>=len(r) or j>=len(ap50) or ap.ndim!=2 or j>=len(ap):
                continue
            item = {
                'epoch':epoch,'class_id':int(cl), 'class_name':CFG['class_names'][int(cl)],
                'precision':float(p[j]),'recall':float(r[j]),
                'AP50':float(ap50[j]), 'AP75':float(ap[j,5]),
                'AP50_95':float(ap[j].mean()),
            }
            RUN_STATE['per_class_epoch_rows'].append(item)
            prefix=f"per_class/{int(cl)}"
            for key in ('precision','recall','AP50','AP75','AP50_95'):
                scalars[f"{prefix}/{key}"]=item[key]
        pd.DataFrame(RUN_STATE['per_class_epoch_rows']).to_csv(REPORTS/'per_class_epoch_history.csv',index=False)

    WB_RUN.log({k:v for k,v in scalars.items() if v is not None}, step=epoch, commit=True)
    RUN_STATE['last_logged_epoch'] = epoch

student.add_callback('on_train_start', ensure_baseline)
student.add_callback('on_train_epoch_start', on_epoch_started)
student.add_callback('on_model_save', on_checkpoint_saved)
student.add_callback('on_fit_epoch_end', on_fit_epoch_logged)

TRAIN_ARGS = {k:CFG[k] for k in PROTOCOL_KEYS if k != 'student_model'}
TRAIN_ARGS.update(data=str(RUNTIME_YAML),device=DEVICE,save=True,
                  project=str(LOCAL_PROJECT_DIR),name=CFG['run_name'],
                  exist_ok=False,verbose=True)
# Not part of the public trainer options: never pass W&B config here.
assert TRAIN_ARGS['save_period']==1
print('W&B artifact collection:',EPOCH_ARTIFACT_NAME)
print('Train args:',TRAIN_ARGS)

## 8 — Train; FAIL LOUDLY if checkpoint upload fails; log run status + metadata

In [ ]:
started = time.perf_counter()
status, error = 'failed', None
torch.cuda.reset_peak_memory_stats(DEVICE)
try:
    train_result = student.train(**TRAIN_ARGS)
    status = 'completed'
except Exception as exc:
    error = f"{type(exc).__name__}: {exc}"
    WB_RUN.summary['research/status'] = 'failed'
    WB_RUN.summary['research/error'] = error
    # Do not call WB_RUN.finish here: keep the run recoverable if a subsequent cell is needed.
    raise
finally:
    RUN_DIR = Path(student.trainer.save_dir).resolve() if getattr(student,'trainer',None) else LOCAL_PROJECT_DIR / CFG['run_name']
    TRAIN_META = {
        'status':status, 'error':error, 'seed':CFG['seed'],
        'duration_seconds_including_artifact_upload':time.perf_counter()-started,
        'artifact_upload_seconds':RUN_STATE['upload_seconds_total'],
        'peak_cuda_allocated_gib':torch.cuda.max_memory_allocated(DEVICE)/2**30,
        'peak_cuda_reserved_gib':torch.cuda.max_memory_reserved(DEVICE)/2**30,
        'actual_batch':RUN_STATE['actual_batch'],
        'completed_epochs':RUN_STATE['last_logged_epoch'],
        'best_epoch':RUN_STATE['best_epoch'],
        'wandb_url':WB_RUN.url,
        'run_dir':str(RUN_DIR),
    }
    (REPORTS/'training_metadata.json').write_text(json.dumps(TRAIN_META,indent=2,ensure_ascii=False),encoding='utf-8')
    for k,v in TRAIN_META.items():
        if v is not None and isinstance(v,(int,float,str,bool)):
            WB_RUN.summary[f'training/{k}'] = v
print('Training metadata:')
print(json.dumps(TRAIN_META,indent=2))

## 9 — Verify checkpoint completeness, compute **best epoch** and full history from `results.csv`

In [ ]:
BEST_PT = RUN_DIR / 'weights/best.pt'
LAST_PT = RUN_DIR / 'weights/last.pt'
RESULTS_CSV = RUN_DIR / 'results.csv'
ARGS_YAML = RUN_DIR / 'args.yaml'
for p in (BEST_PT,LAST_PT,RESULTS_CSV,ARGS_YAML):
    assert p.is_file(), f'Missing required output: {p}'

history = pd.read_csv(RESULTS_CSV)
history.columns = history.columns.str.strip()
assert len(history) == RUN_STATE['last_logged_epoch'] == RUN_STATE['last_saved_epoch'], (
    f"Epoch mismatch: CSV={len(history)} W&B={RUN_STATE['last_logged_epoch']} saved={RUN_STATE['last_saved_epoch']}"
)
assert len(RUN_STATE['artifact_versions']) == len(history), 'An epoch checkpoint was not uploaded.'
assert 'metrics/mAP50-95(B)' in history.columns
assert np.isfinite(history['metrics/mAP50-95(B)'].to_numpy()).all()
assert not any('distill' in c.lower() or 'dis_loss' in c.lower() for c in history.columns)

ckpt_manifest = pd.read_csv(REPORTS/'checkpoint_manifest.csv')
assert len(ckpt_manifest)==len(history)
assert ckpt_manifest['artifact_version'].notna().all()
# Ultralytics detection fitness uses mAP50-95; explicit best tracking is more accurate than rounded CSV.
assert RUN_STATE['best_epoch'] is not None
best_row = history.loc[history['epoch']==RUN_STATE['best_epoch']]
assert len(best_row)==1
saved_config = yaml.safe_load(ARGS_YAML.read_text())
for k in PROTOCOL_KEYS:
    if k != 'student_model':
        assert saved_config[k] == CFG[k],f'Protocol mismatch {k}: {saved_config[k]} != {CFG[k]}'

summary_training = {
    'best_epoch':int(RUN_STATE['best_epoch']),
    'last_epoch':int(history.epoch.iloc[-1]),
    'epochs_completed':int(len(history)),
    'best_val_mAP50_95_from_csv':float(best_row['metrics/mAP50-95(B)'].iloc[0]),
    'last_val_mAP50_95_from_csv':float(history['metrics/mAP50-95(B)'].iloc[-1]),
    'early_stopped':bool(len(history)<CFG['epochs']),
    'checkpoint_versions':len(RUN_STATE['artifact_versions']),
}
(REPORTS/'training_summary.json').write_text(json.dumps(summary_training,indent=2),encoding='utf-8')
for k,v in summary_training.items(): WB_RUN.summary['training/'+k]=v
WB_RUN.log({
    'training/epoch_history_table':wandb.Table(dataframe=history),
    'training/per_class_epoch_history_table':wandb.Table(dataframe=pd.read_csv(REPORTS/'per_class_epoch_history.csv'))
}, commit=True)

print(json.dumps(summary_training,indent=2))
print('W&B per-epoch collection:',EPOCH_ARTIFACT_NAME)
assert (REPORTS/'per_class_epoch_history.csv').exists(), 'Per-class epoch history not generated.'
print('Best alias:', EPOCH_ARTIFACT_NAME+':best')
display(history.tail())
display(ckpt_manifest.tail())

## 10 — Evaluate **best.pt** on validation split only (NOT test)

Capture aggregate precision/recall/mAP50/mAP50-95; per-class precision/recall/AP50/AP75/AP50-95; confusion matrix/PR curves from Ultralytics. Keep training-run validation metrics distinct from this explicit best-checkpoint re-evaluation.

In [ ]:
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(RUNTIME_YAML),split='val',imgsz=CFG['imgsz'],
    batch=CFG['batch'],device=DEVICE,workers=CFG['workers'],
    plots=True,save_json=False,verbose=True,
    project=str(WORK_ROOT/'validation'),name='best-val',exist_ok=False,
)
VAL_DIR = Path(val_metrics.save_dir)
box = val_metrics.box
indices = np.asarray(box.ap_class_index, dtype=int)

def keyed_per_class(values):
    arr=np.asarray(values,dtype=float).reshape(-1)
    return {int(cls):float(arr[i]) for i,cls in enumerate(indices) if i < len(arr)}

class_p=keyed_per_class(box.p)
class_r=keyed_per_class(box.r)
class_ap50=keyed_per_class(box.ap50)
class_ap5095=keyed_per_class(box.ap.mean(axis=1))
class_ap75=keyed_per_class(box.ap[:,5])  # IoU thresholds 0.50,0.55,...0.95
PER_CLASS = pd.DataFrame([{
    'class_id':i,'class_name':name,
    'precision':class_p.get(i,np.nan),'recall':class_r.get(i,np.nan),
    'AP50':class_ap50.get(i,np.nan),'AP75':class_ap75.get(i,np.nan),
    'AP50_95':class_ap5095.get(i,np.nan),
} for i,name in enumerate(CFG['class_names'])])
PER_CLASS.to_csv(REPORTS/'per_class_val.csv',index=False)

CHECKPOINT_AUDIT = {}
for label, path in [('best',BEST_PT),('last',LAST_PT)]:
    reload = YOLO(str(path))
    CHECKPOINT_AUDIT[label]={
        'size_mb':path.stat().st_size/1e6,
        'parameters':int(sum(p.numel() for p in reload.model.parameters())),
        'nc':int(reload.model.model[-1].nc),
        'has_teacher_model':bool(hasattr(reload.model,'teacher_model')),
        'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
    }
    assert CHECKPOINT_AUDIT[label]['nc']==5
    assert not CHECKPOINT_AUDIT[label]['has_teacher_model']
    del reload
(REPORTS/'checkpoint_audit.json').write_text(json.dumps(CHECKPOINT_AUDIT,indent=2),encoding='utf-8')

VALIDATION = {
    'split':'val', 'test_evaluated':False,
    'precision':float(box.mp),'recall':float(box.mr),
    'mAP50':float(box.map50), 'mAP50_95':float(box.map),
    'best_epoch':summary_training['best_epoch'],
    'dataset_fingerprint':DATA_FINGERPRINT,
    'best_pt_size_mb':CHECKPOINT_AUDIT['best']['size_mb'],
    'best_pt_parameters':CHECKPOINT_AUDIT['best']['parameters'],
}
(REPORTS/'best_val_summary.json').write_text(json.dumps(VALIDATION,indent=2),encoding='utf-8')
for k,v in VALIDATION.items():
    if v is not None and isinstance(v,(int,float,bool,str)):
        WB_RUN.summary['validation/'+k]=v
WB_RUN.log({
    'evaluation/best_per_class':wandb.Table(dataframe=PER_CLASS),
    'evaluation/best_aggregate':wandb.Table(dataframe=pd.DataFrame([VALIDATION])),
},commit=True)

# Native Ultralytics saves e.g. confusion_matrix.png, *_curve.png, val_batch*.jpg
plot_payload={}
for p in sorted(VAL_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
        plot_payload[f'validation_plots/{p.stem}']=wandb.Image(str(p))
if plot_payload: WB_RUN.log(plot_payload,commit=True)
# Include standard training diagnostics: loss curves/results.png, class plots, train batches.
train_plot_payload={}
for p in sorted(RUN_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
        train_plot_payload[f'training_plots/{p.stem}']=wandb.Image(str(p))
if train_plot_payload: WB_RUN.log(train_plot_payload,commit=True)

display(PER_CLASS)
print(json.dumps(VALIDATION,indent=2))
print('Validation plots in:',VAL_DIR)
del best_model
gc.collect()
torch.cuda.empty_cache()

## 11 — Profiling of **best.pt** (separate from training metric semantics)

In [ ]:
# This is optional hardware benchmark on batch=1 GPU, full predict() path;
# it is not a standardized final deployment benchmark. Use same GPU/hardware for all comparisons.
from statistics import median

profile_model=YOLO(str(BEST_PT))
valid_images=sorted(p for p in SPLIT_IMAGES['val'].iterdir() if p.suffix.lower() in IMAGE_EXT)
assert valid_images
img_path=str(valid_images[0])
times_ms=[]
try:
    for _ in range(5):
        _=profile_model.predict(img_path,imgsz=CFG['imgsz'],device=DEVICE,verbose=False,save=False)
    for _ in range(30):
        torch.cuda.synchronize(DEVICE)
        t0=time.perf_counter()
        _=profile_model.predict(img_path,imgsz=CFG['imgsz'],device=DEVICE,verbose=False,save=False)
        torch.cuda.synchronize(DEVICE)
        times_ms.append((time.perf_counter()-t0)*1000)
    PROFILE={
        'bench_source':'single validation image, 5 warmups + 30 timed YOLO.predict calls',
        'device':ENV['gpu'],'batch':1,'imgsz':CFG['imgsz'],
        'median_wall_ms':float(median(times_ms)),
        'mean_wall_ms':float(np.mean(times_ms)),
        'p95_wall_ms':float(np.percentile(times_ms,95)),
        'fps_inverse_median_wall_ms':float(1000/median(times_ms)),
        'note':'Includes Python I/O, preprocessing, forward, NMS; not pure neural inference or final deployment benchmark',
    }
except Exception as exc:
    PROFILE={'status':'failed','error':repr(exc)}
(REPORTS/'pilot_walltime_profile.json').write_text(json.dumps(PROFILE,indent=2),encoding='utf-8')
for k,v in PROFILE.items():
    if isinstance(v,(int,float,str)):
        WB_RUN.summary['pilot_profile/'+k]=v
print(PROFILE)
del profile_model
gc.collect()
torch.cuda.empty_cache()

## 12 — Final **separate** artifact: training history, model checkpoints, reports, validation figures

In [ ]:
FINAL_NAME = f"inatrc-yolo26n-baseline-final-{WB_RUN.id}"
final = wandb.Artifact(FINAL_NAME, type='model',metadata={
    'seed':CFG['seed'],'best_epoch':int(RUN_STATE['best_epoch']),
    'epochs_completed':int(len(history)), 'validation_mAP50_95':float(box.map),
    'dataset_fingerprint':DATA_FINGERPRINT,'test_evaluated':False,
})
for p,name in ((BEST_PT,'best.pt'),(LAST_PT,'last.pt'),(RESULTS_CSV,'results.csv'),
               (ARGS_YAML,'args.yaml'),(RUNTIME_YAML,'data_runtime.yaml')):
    final.add_file(str(p),name=name)
for p in sorted(REPORTS.rglob('*')):
    if p.is_file(): final.add_file(str(p),name='reports/'+str(p.relative_to(REPORTS)))
for p in sorted(VAL_DIR.rglob('*')):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.json'}:
        final.add_file(str(p),name='validation/'+str(p.relative_to(VAL_DIR)))
for p in sorted(RUN_DIR.iterdir()):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
        final.add_file(str(p),name='training_plots/'+p.name)

uploaded_final=WB_RUN.log_artifact(final,aliases=['final','best-checkpoint',f"seed-{CFG['seed']}"])
uploaded_final.wait()
WB_RUN.summary['checkpoint/final_artifact'] = FINAL_NAME
WB_RUN.summary['research/status'] = 'completed'
WB_RUN.summary['research/test_evaluated'] = False

print('Final artifact:',FINAL_NAME,'version:',uploaded_final.version)
print('All-epochs artifact:',EPOCH_ARTIFACT_NAME)
print('Best epoch:',RUN_STATE['best_epoch'])
print('Epoch versions logged:',len(RUN_STATE['artifact_versions']))
print('W&B run:',WB_RUN.url)
WB_RUN.finish()

ZIP_PATH=WORK_ROOT/f"{CFG['run_name']}-FINAL.zip"
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in (BEST_PT,LAST_PT,RESULTS_CSV,ARGS_YAML,RUNTIME_YAML):
        z.write(p,arcname=p.name)
    for p in REPORTS.rglob('*'):
        if p.is_file(): z.write(p,arcname='reports/'+str(p.relative_to(REPORTS)))
    for p in VAL_DIR.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.json'}:
            z.write(p,arcname='validation/'+str(p.relative_to(VAL_DIR)))
    for p in RUN_DIR.iterdir():
        if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
            z.write(p,arcname='training_plots/'+p.name)
print('Kaggle bundle:',ZIP_PATH)
display(FileLink(str(ZIP_PATH)))

## 13 — W&B audit checklist (after the run finishes)

Open your run and verify:

1. **Charts / History:** `train/*`, `val/*`, `metrics/precision(B)`, `metrics/recall(B)`, `metrics/mAP50(B)`, `metrics/mAP50-95(B)`, `lr/*`, `fit/fitness`, `fit/is_current_best`, `system/gpu_memory_*` each epoch.
2. **Summary:** `training/best_epoch`, `training/last_epoch`, `training/early_stopped`, `training/checkpoint_versions`, `training/duration_seconds_including_artifact_upload`, `validation/mAP50_95`, `research/test_evaluated=False`.
3. **Artifacts (model):** `inatrc-yolo26n-baseline-epochs-<run_id>` contains **one version per epoch**, `epoch-001`, ..., `best` alias; `latest` means latest *artifact version*, not best model.
4. **Artifacts (final):** `inatrc-yolo26n-baseline-final-<run_id>` contains `best.pt`, `last.pt`, `results.csv`, `args.yaml`, `data_runtime.yaml`, `reports/*`, `validation/*`.
5. **Tables:** dataset audit, class distribution, full CSV history, per-class precision, recall, AP50, AP75, AP50-95, aggregate validation.
6. **Validation visuals:** confusion matrices, F1/PR/P/R curves, sample predictions **if created by Ultralytics**.
7. **Research integrity:** `test` was *not evaluated*. Log technical failures and do not silently relax batch/seed/hyperparameters. Test leakage may still arise through near-duplicate CCTV frames, which this notebook does not assess.

### Interpret the training/checkpoint numbers
- `save_period=1` **saves every epoch**. Ultralytics names it `epoch0.pt` for epoch #1, but W&B uses the human-friendly `epoch-001` alias.
- **Best epoch** is taken from the live trainer fitness comparison (not rounded CSV), and the corresponding W&B version has `best` alias.
- W&B per-epoch model uploads add wall-time and potentially substantial quota usage; do not equate `duration_seconds_including_artifact_upload` with pure optimizer training time.
- Final `best.pt` may have its embedded `epoch=-1` because the optimizer is stripped. **The saved `training/best_epoch` field and `checkpoint_manifest.csv` are the reliable record.**